# 19 — Intermediate Review and Readiness Assessment

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/19', {1: 'a8718e3b2c03de13', 2: '3ef0e432d9747ac4', 3: 'ddc619d3b36b02d9', 4: 'ba116e11c5e826a1', 5: '1add09793ce10c0d', 6: '01367ea8bba59dcf', 7: 'cac080ab43e2ef44'})

## 1. Models and validation

- Model records with a `@dataclass`; validate in `__post_init__` and raise `ValueError`.
- Use `field(default_factory=list)` for mutable defaults.
- `@property` computes derived values; `frozen=True` makes records read-only.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Order:
    customer: str
    items: list = field(default_factory=list)
    def __post_init__(self):
        if not self.customer.strip():
            raise ValueError("customer is required")
    @property
    def total(self):
        return sum(price for _, price in self.items)

o = Order("Ana", [("pen", 1.5), ("ink", 3.0)])
print(o.total)

**Exercise 1.** Compute the order total from the items.

In [ ]:
items = [("pen", 1.5), ("ink", 3.0), ("pad", 2.5)]
total = sum(___ for _, price in items)
check(1, total)

**Exercise 2.** Two orders created with `default_factory=list`: do they share one list?

In [ ]:
predicted = ___   # True or False
check(2, predicted)

## 2. Parse, then analyse

- Keep stages separate: parse text → validate → aggregate. Each is a small, testable function.
- Generators keep big inputs lazy; `sorted`, `groupby`, `Counter` do the summarising.
- Reject bad rows with a count, rather than crashing or silently ignoring them.

In [ ]:
from collections import Counter
lines = ["pen,2", "ink,x", "pen,1", "pad,4"]
good, bad = [], 0
for line in lines:
    name, qty = line.split(",")
    try:
        good.append((name, int(qty)))
    except ValueError:
        bad += 1
totals = Counter()
for name, qty in good:
    totals[name] += qty
print(dict(totals), "rejected:", bad)

**Exercise 3.** Parse `name,qty` lines into tuples with an int quantity.

In [ ]:
lines = ["pen,2", "pad,4"]
rows = [(name, ___) for name, qty in (line.split(",") for line in lines)]
check(3, rows)

**Exercise 4.** Total per name with `Counter`.

In [ ]:
from collections import Counter
rows = [("pen", 2), ("pad", 4), ("pen", 1)]
totals = Counter()
for name, qty in rows:
    totals[name] += ___
check(4, dict(totals))

## 3. Store and test

- Use `sqlite3.connect(":memory:")` in tests so nothing is left behind.
- Use `?` placeholders and `with conn:` transactions.
- Test behaviour (what goes in, what comes out), not internal details; mock the network and the clock.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
with conn:
    conn.execute("CREATE TABLE stock (name TEXT PRIMARY KEY, qty INTEGER)")
    conn.executemany("INSERT INTO stock VALUES (?, ?)", [("pen", 3), ("pad", 4)])
print(conn.execute("SELECT SUM(qty) FROM stock").fetchone()[0])

**Exercise 5.** Sum the stock with SQL.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE stock (name TEXT, qty INTEGER)")
conn.executemany("INSERT INTO stock VALUES (?, ?)", [("pen", 3), ("pad", 4), ("ink", 1)])
total = conn.execute("SELECT SUM(qty) FROM stock").fetchone()[___]
check(5, total)

**Exercise 6.** Which placeholder keeps user text out of the SQL itself? Write it.

In [ ]:
answer = ___
check(6, answer)

**Exercise 7.** A mock clock returns 25. What does `is_expired(now=clock, deadline=20)` give?

In [ ]:
from unittest.mock import Mock
clock = Mock(return_value=25)

def is_expired(now, deadline):
    return now() > deadline

predicted = ___
check(7, predicted)

## Independent build

Implement reconcile(expected, received) for Order Desk. Each input maps SKU to nonnegative units. Return only nonzero differences (received minus expected), treating absent SKUs as zero. Preserve both inputs. Explain how a caller can distinguish shortages from surpluses.

In [ ]:
# Write your implementation here.

In [ ]:
if 'reconcile' not in globals():
    print("Implement reconcile, then rerun this check.")
else:
    left = {'A': 2, 'B': 1}
    right = {'B': 1, 'C': 3}
    assert reconcile(left, right) == {'A': -2, 'C': 3}
    assert left == {'A': 2, 'B': 1} and right == {'B': 1, 'C': 3}
    assert reconcile({}, {}) == {}
    print("Acceptance examples passed.")

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Where should validation of external data happen?

   - **a)** At the boundary, when the data enters
   - **b)** Deep inside every function
   - **c)** Never

**2.** Domain logic that doesn't touch files or network is easier to…

   - **a)** Hide
   - **b)** Deploy
   - **c)** Test

**3.** Before Advanced, you should be able to…

   - **a)** Avoid classes
   - **b)** Explain and test your design choices
   - **c)** Memorize the standard library

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/19', {1: 'f7420d063e6f', 2: '9bb5ca1d81e1', 3: 'b2ba4d35d6ae'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** the [intermediate projects](../projects/) and [exercise workbooks](../exercises/)